<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Filter Sites by Available Resources

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook shows you how to use **filter functions** to find FABRIC sites that match specific resource requirements. Instead of manually scanning a large table of sites, you can write lambda expressions to programmatically select sites with the exact hardware you need -- such as available SmartNICs, GPUs, geographic location, or PTP capability.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Use `filter_function` with `list_sites()` to filter sites by resource availability
2. Combine multiple filter conditions (e.g., hardware **and** geography)
3. Display only specific columns alongside filters using the `fields` parameter
4. Filter sites by special capabilities like PTP (Precision Time Protocol)
5. Use `get_random_site()` and `get_random_sites()` to programmatically select sites matching your criteria

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must** complete the environment setup:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Familiarize yourself with [list_all_resources](./list_all_resources.ipynb) to understand site fields

**Tip:** Run `fablib.list_sites(pretty_names=False)` to see the programmatic field names you can use in filter functions.

</div>

## Background: How Filter Functions Work

Many FABlib `list_xxx()` and `get_random_site()` methods accept a `filter_function` argument. This is a **callable** (typically a lambda) that receives a dictionary representing one site and returns `True` or `False`.

```
filter_function receives a dict like:
{
    'name': 'RENC',
    'cores_available': 120,
    'ram_available': 384,
    'nic_connectx_5_available': 4,
    'nic_connectx_6_available': 2,
    'location': (35.78, -78.64),     # (latitude, longitude)
    'ptp_capable': True,
    'rtx6000_available': 2,
    ...                               # many more fields
}
```

Only sites where the function returns `True` are included in the results.

Although this example focuses on `fablib.list_sites()`, all FABlib `list_xxx()` methods accept the `fields` argument. The specific fields available differ for each object type. Calling the list method without setting `fields` shows all available fields.

---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Filter Sites by a Single Condition

The simplest filter checks a single field. Below, we find all sites that have **more than 2 ConnectX-5 SmartNICs** available. The lambda function receives a dictionary `x` representing each site and returns `True` only if the condition is met.

In [ ]:
# Filter: show only sites with more than 2 ConnectX-5 SmartNICs available
# The lambda receives a dict 'x' with all site fields; return True to include the site
fablib.list_sites(filter_function=lambda x: x['nic_connectx_5_available'] > 2);

## Step 3: Combine Multiple Filter Conditions

You can combine conditions using Python's `and` / `or` operators. The following example finds sites with:
- More than 2 ConnectX-5 SmartNICs available, **AND**
- Located **west** of St. Louis, MO (longitude less than St. Louis)

Each site has a `location` field containing a `(latitude, longitude)` tuple. Longitude values decrease as you move west in the US.

In [ ]:
# Define a geographic reference point (St. Louis, MO coordinates)
st_louis_lat_long=(32.773081, -96.797448)

# Filter: ConnectX-5 available > 2 AND site is west of St. Louis (lower longitude)
fablib.list_sites(filter_function=lambda x: x['nic_connectx_5_available'] > 2 and x['location'][1] < st_louis_lat_long[1]);

## Step 4: Combine Filters with Field Selection

You can use `filter_function` and `fields` together to both filter the rows (sites) and limit the columns displayed. This gives you a compact, focused view.

In [ ]:
# Same geographic reference point
st_louis_lat_long=(32.773081, -96.797448)

# Filter sites AND select only specific columns to display
fablib.list_sites(filter_function=lambda x: x['nic_connectx_5_available'] > 2 and x['location'][1] < st_louis_lat_long[1],
                      fields=['name','address', 'nic_connectx_5_available']);

## Step 5: Filter by Special Capabilities (PTP)

Some FABRIC sites support **Precision Time Protocol (PTP)**, which provides nanosecond-level clock synchronization. If your experiment requires precise timing, filter for PTP-capable sites.

In [ ]:
# Filter: show only sites that support Precision Time Protocol (PTP)
fablib.list_sites(filter_function=lambda x: x['ptp_capable'] is True)

## Step 6: Get a Random Site Matching Criteria

When writing automated scripts, you often want to **pick a site** that meets your requirements rather than listing all matches. The `fablib.get_random_site()` method returns a single site name (string) from among those matching your filter.

The example below finds one site in the **eastern US** and one in the **western US**, both with available ConnectX-6 SmartNICs and PTP support.

<div class="fab-danger">

**Important:** Because the filter checks for *currently available* resources, it may return `None` if no site matches at this moment. Always check the return value before using it.

</div>

In [ ]:
# Geographic reference point for east/west split
st_louis_lat_long=(32.773081, -96.797448)

# Get one random site WEST of St. Louis with ConnectX-6 and PTP
west_site = fablib.get_random_site(filter_function=lambda x: x['nic_connectx_6_available'] > 0 and x['location'][1] < st_louis_lat_long[1] and x['ptp_capable'] is True)                                                                                                                                                                                                                           

# Get one random site EAST of St. Louis with ConnectX-6 and PTP
east_site = fablib.get_random_site(filter_function=lambda x: x['nic_connectx_6_available'] > 0 and x['location'][1] > st_louis_lat_long[1] and x['ptp_capable'] is True)                                                                                                                                                                                                                           

print(f"west_site: {west_site}")
print(f"east_site: {east_site}")

## Step 7: Get Multiple Random Sites

Use `fablib.get_random_sites(count=N)` to get a list of `N` distinct sites matching your filter. This is useful for multi-site experiments where you need several geographically distributed nodes.

In [ ]:
# Get 4 random sites that each have more than 2 RTX6000 GPUs available
sites = fablib.get_random_sites(count=4, filter_function=lambda x: x['rtx6000_available'] > 2)                                                                                                                                                                                                                           

print(f"sites: {sites}")

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `get_random_site()` returns `None` | No site currently matches your filter | Relax your filter criteria or try again later |
| `KeyError` in filter function | Field name is misspelled | Run `list_sites(pretty_names=False)` to see valid field names |
| Filter returns no sites | Resources are fully allocated | Check current availability with `list_sites()` first |
| Geographic filter gives unexpected results | Latitude/longitude confused | `location` is `(lat, lon)`; longitude is index `[1]` |
| `get_random_sites(count=4)` returns fewer than 4 | Not enough sites match your criteria | Reduce `count` or relax filter conditions |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.list_sites()` | List sites with optional filtering and field selection | [list_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_sites) |
| `fablib.get_random_site()` | Get one random site matching a filter | [get_random_site](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_site) |
| `fablib.get_random_sites()` | Get multiple random sites matching a filter | [get_random_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_sites) |

## What's Next?

Now that you can find sites matching your requirements, explore these notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **List All Resources** | [list_all_resources](./list_all_resources.ipynb) | Explore all output formats and time-based queries |
| **Hello FABRIC** | [hello_fabric](../hello_fabric/hello_fabric.ipynb) | Create your first experiment |
| **Customizing Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, disk, and OS image |
| **Advanced Scheduling** | [advanced_scheduling_slice](../create_slice/advanced_scheduling_slice.ipynb) | Reserve resources for a future time window |
| **SmartNICs** | [create_l2network_wide_area](../create_l2network/create_l2network_wide_area.ipynb) | Use ConnectX SmartNICs for high-performance networking |